**Duplicates and noise:**
- Inflate confidence
- Bias models
- Break train-test independence
- Cause data leakage

In [27]:
import pandas as pd

df = pd.DataFrame({
     "user_id": [1, 2, 3, 3, 4, 5, 5],
    "city": ["Mumbai", "Delhi", "delhi ", "Delhi", "Bangalore", "Mumbai", "mumbai"],
    "income": [50000, 60000, 65000, 65000, 70000, 50000, 50000],
    "purchased": [0, 1, 1, 1, 0, 0, 0]
})

### Exact Duplicate Rows
#### Detect

In [21]:
df.duplicated().sum()

np.int64(0)

#### View duplicates

In [22]:
df[df.duplicated()]

,user_id,city,income,purchased


#### Remove
Safe **only if** duplicates are accidental.

In [23]:
df = df.drop_duplicates()

### Duplicate Entities
Same entity repeated with same or conflicting data.    
**Example: Same `user_id`**

In [24]:
df[df.duplicated(subset=["user_id"], keep=False)]

,user_id,city,income,purchased
2,3,delhi,65000,1
3,3,Delhi,65000,1
5,5,Mumbai,50000,0
6,5,mumbai,50000,0


#### How to Handle
**Case 1: Identical records -> keep one**

In [25]:
df = df.drop_duplicates(subset=["user_id"])
df

,user_id,city,income,purchased
0,1,Mumbai,50000,0
1,2,Delhi,60000,1
2,3,delhi,65000,1
4,4,Bangalore,70000,0
5,5,Mumbai,50000,0


**Case 2: Aggregation needed**

In [28]:
df = df.groupby("user_id").agg({
    "income": "mean",
    "purchased": "max",
    "city": "first"
}).reset_index()

df

,user_id,income,purchased,city
0,1,50000.0,0,Mumbai
1,2,60000.0,1,Delhi
2,3,65000.0,1,delhi
3,4,70000.0,0,Bangalore
4,5,50000.0,0,Mumbai


### Noisy Categorical Data
#### Detect Inconsistencies

In [29]:
df["city"].value_counts()

city
Mumbai       2
Delhi        1
delhi        1
Bangalore    1
Name: count, dtype: int64

#### Normal Categories

In [32]:
df["city"] = (
    df["city"]
    .str.strip()
    .str.lower()
)

df["city"], df["city"].value_counts()

(0       mumbai
 1        delhi
 2        delhi
 3    bangalore
 4       mumbai
 Name: city, dtype: object,
 city
 mumbai       2
 delhi        2
 bangalore    1
 Name: count, dtype: int64)

#### Rare Category Handling

In [35]:
city_counts = df["city"].value_counts()
rare = city_counts[city_counts < 2].index

df["city"] = df["city"].replace(rare, "Other")
df["city"]

0    mumbai
1     delhi
2     delhi
3     Other
4    mumbai
Name: city, dtype: object

Prevents one-hot explosion.

### Conflicting Labels
Same features -> different target.

In [36]:
df[df.duplicated(subset=["user_id", "income"], keep=False)]

,user_id,income,purchased,city


This indicates:
- Data error
- Label noise
- Logging bugs

### Textual Noise

In [38]:
df["city"] = df["city"].str.replace(r"[^a-zA-Z ]", "", regex=True)
df["city"]

0    mumbai
1     delhi
2     delhi
3     Other
4    mumbai
Name: city, dtype: object

### Train-Test Contamination Check

In [39]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2)

set(train["user_id"]).intersection(set(test["user_id"]))

set()

Same entity in train & test -> inflated accuracy.